# 28 — Especialista PONDERADO para vendedores sin historial

**Idea propia.** Corregir el problema de I4 (Fase I): aquel especialista se entrenó **solo** con vendedores de ≤ 1 publicación (~10.000 filas por fold), perdió los ejemplos del resto y quedó **peor** que el modelo final en su propio grupo (0,8825 contra 0,8967).

**Diferencia con I4.** El especialista se entrena con **todo** el train del fold, pero las filas de vendedores con **1 sola publicación** (en el train del fold) pesan `w` (`sample_weight`) y el resto 1. Es como si esas filas aparecieran `w` veces: el modelo sigue aprendiendo los patrones generales, pero al elegir cada corte del árbol le importa más acertar en filas que se parecen a un vendedor nuevo. Es la idea de AdaBoost (`material/Arboles/Árboles.pdf`, slide 62: "darle mayor peso a las muestras"), pero con el peso fijado de antemano por grupo, no por error.

**Por qué las filas con 1 publicación (y no las "sin historial").** Un vendedor de validación sin historial no tiene filas en el train del fold. Lo más parecido en el train son los vendedores con 1 sola publicación: el `TargetEncoder` hace *cross-fitting*, así que al codificar esa fila no se ve a sí misma y el vendedor queda como desconocido, igual que en el test.

**Por qué no es leakage.**
1. El peso sale del **conteo de publicaciones del vendedor en el train del fold**: no usa `condition` ni mira la validación.
2. Todo lo que aprende de los datos sigue adentro del `Pipeline`, ajustado solo con el train del fold.
3. `w`, el grupo de variables y `α` se eligen con 4 folds y se evalúan en el quinto, rotando: las 90 variantes no inflan el resultado.
4. Detalle: las OOF de los 4 folds de elección salen de modelos que vieron el quinto fold al entrenar (igual que en I2, I4 y 27-4a). Se eligen solo 3 números, así que el efecto es chico.

**Partes:**
1. Especialista = HistGB P7a + duración con `sample_weight`, w ∈ {2, 3, 5}.
2. Ablación por grupos (solo para el especialista): sin texto, sin vendedor, sin fotos, sin fechas, sin precio.
3. Uso: en las filas sin historial, (1 − α)·P7c + α·especialista, α ∈ {0; 0,25; 0,5; 0,75; 1}; el resto queda con P7c.
4. Elección anidada (4 folds → quinto) y comparación pareada contra P7c + duración (0,9420); semilla 7 si pasa.

**Material:**
- pesos por fila: AdaBoost, `material/Arboles/Árboles.pdf`, slides 62-63 (el `sample_weight` como parámetro **no está en el material**);
- selección de variables: Lasso, `Continuación de regresion y Regulización L1 y L2.pdf`, slide 36, y *random feature selection*, SIA `S8 - Ensamble.pdf`, slide 17. **La ablación por grupos no está en el material**: se saca un grupo entero, se reentrena y se mide;
- promedio ponderado de probabilidades (voting soft): SIA `S8 - Ensamble.pdf`;
- target encoder: `Copia de AP_Categóricas.ipynb`.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, COLUMNAS_COMBOS)

pd.set_option("display.width", 200)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"

oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
assert (oof_ref["fold"].to_numpy() == folds).all()
p_hgb = oof_ref["histgb"].to_numpy()
p_final = 0.5 * p_hgb + 0.5 * oof_ref["catboost"].to_numpy()


def acc_por_fold(prob, folds_, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds_[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


def calcular_sin_historial(folds_):
    # Vendedores "sin historial" en cada fold: no tienen ninguna publicación en el train de ese fold
    sh = np.zeros(n, bool)
    for k in range(1, 6):
        va = folds_ == k
        sh[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()
    return sh


sin_historial = calcular_sin_historial(folds)
acc_final = acc_por_fold(p_final, folds)
print("Modelo final (P7c + duración):", acc_final.mean().round(4),
      "| sin historial:", acc_por_fold(p_final, folds, sin_historial).mean().round(4), f"({sin_historial.mean():.1%} de las filas)")
print("HistGB P7a + duración (w = 1):", acc_por_fold(p_hgb, folds).mean().round(4),
      "| sin historial:", acc_por_fold(p_hgb, folds, sin_historial).mean().round(4))

Modelo final (P7c + duración): 0.942 | sin historial: 0.8931 (30.7% de las filas)
HistGB P7a + duración (w = 1): 0.9412 | sin historial: 0.8923


## 1-2. Especialista ponderado y ablación por grupos

**Qué hace `armar_especialista(grupo)`:** arma el HistGB P7a + duración (`armar_pipeline_histgb_combos()`, con el modelo de texto cacheado) y le saca las columnas del grupo:

| Grupo | Qué se saca |
|---|---|
| `ninguno` | nada (P7a completo) |
| `texto` | `tfidf` (500 columnas) + `prob_texto_new` |
| `vendedor` | `seller_id` del TE, los 7 combos `c_seller_*` y `VendedorEncoder` |
| `fotos` | `n_fotos` + las 4 `foto_*` |
| `fechas` | `hora_inicio`, `gap_actualizacion` y `duracion_dias` |
| `precio` | `log_price`, `c_cat_price` y `c_seller_price` (el precio promedio del vendedor queda en `VendedorEncoder`) |

**Qué hace `ajustar_fold(grupo, w, k)`:** toma el train del fold `k`, cuenta las publicaciones de cada vendedor **en ese train**, le da peso `w` a las filas de vendedores con 1 sola y 1 al resto, entrena con `modelo__sample_weight` (solo el HistGB usa el peso; el TF-IDF, el modelo de texto y el `TargetEncoder` no) y predice el fold de validación.

Las 18 variantes (3 pesos × 6 grupos) se guardan en `oof_especialista.csv` para no recalcularlas.

In [2]:
SACAR = {
    "ninguno": [],
    "texto": [],
    "vendedor": ["seller_id"] + [c for c in COLUMNAS_COMBOS if c.startswith("c_seller_")],
    "fotos": ["n_fotos", "foto_log_px_primera", "foto_pct_cuadradas", "foto_pct_camara", "foto_n_tamanios"],
    "fechas": ["hora_inicio", "gap_actualizacion"],
    "precio": ["log_price", "c_cat_price", "c_seller_price"],
}
SACAR_TRANSFORMERS = {"texto": ["tfidf", "texto"], "vendedor": ["vendedor_simple"], "fechas": ["duracion"]}
PESOS = [2, 3, 5]
GRUPOS = list(SACAR)


def armar_especialista(grupo):
    pipeline = armar_pipeline_histgb_combos()
    nuevos = []
    for nombre, transformer, columnas in pipeline.named_steps["preprocesador"].transformers:
        if nombre in SACAR_TRANSFORMERS.get(grupo, []):
            continue
        if nombre == "texto":     # modelo de texto cacheado (mismo resultado, sin reentrenarlo cada vez)
            transformer = Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)), ("identidad", "passthrough")],
                                   memory=Memory(carpeta_lab, verbose=0))
        elif nombre != "vendedor_simple" and isinstance(columnas, list):
            columnas = [c for c in columnas if c not in SACAR[grupo]]
        nuevos.append((nombre, transformer, columnas))
    pipeline.named_steps["preprocesador"].transformers = nuevos
    return pipeline


def ajustar_fold(grupo, w, k, folds_):
    entrena, valida = folds_ != k, folds_ == k
    X_entrena = train[entrena]
    publicaciones = X_entrena["seller_id"].map(X_entrena["seller_id"].value_counts())
    peso = np.where(publicaciones == 1, float(w), 1.0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo = armar_especialista(grupo).fit(X_entrena, y_np[entrena], modelo__sample_weight=peso)
    return k, modelo.predict_proba(train[valida])[:, 1]


def oof_especialistas(folds_, archivo):
    guardado = pd.read_csv(archivo) if os.path.exists(archivo) else pd.DataFrame({"fold": folds_})
    assert (guardado["fold"].to_numpy() == folds_).all()
    for w in PESOS:
        for grupo in GRUPOS:
            col = f"w{w}_{grupo}"
            if col in guardado:
                continue
            inicio = time.time()
            prob = np.empty(n)
            for k, p in Parallel(n_jobs=5)(delayed(ajustar_fold)(grupo, w, k, folds_) for k in range(1, 6)):
                prob[folds_ == k] = p
            guardado[col] = prob
            guardado.to_csv(archivo, index=False)
            print(f"{col:16s} {(time.time() - inicio) / 60:.1f} min", flush=True)
    return {c: guardado[c].to_numpy() for c in guardado if c != "fold"}


especialistas = oof_especialistas(folds, os.path.join(carpeta_lab, "oof_especialista.csv"))

w2_ninguno       0.8 min


w2_texto         0.2 min


w2_vendedor      0.7 min


w2_fotos         0.7 min


w2_fechas        0.8 min


w2_precio        0.7 min


... (8 outputs left out)

w5_vendedor      0.7 min


w5_fotos         0.7 min


w5_fechas        0.7 min


w5_precio        0.7 min


### Tabla descriptiva (todas las variantes con todo el OOF)

Para cada peso y grupo: la accuracy del especialista **solo** en vendedores sin historial (la ablación) y en total, y la accuracy total del promedio con P7c para cada α. **Es descriptiva:** elegir el máximo de esta tabla sería optimista (90 variantes); la medición honesta es la elección anidada de la sección 3.

In [3]:
ALFAS = [0, 0.25, 0.5, 0.75, 1]


def combinar(p_base, p_esp, sh, alfa):
    return np.where(sh, (1 - alfa) * p_base + alfa * p_esp, p_base)


filas = [{"w": 1, "grupo": "ninguno (P7a)", "esp_sin_hist": acc_por_fold(p_hgb, folds, sin_historial).mean(),
          "esp_total": acc_por_fold(p_hgb, folds).mean(),
          **{f"α={a}": acc_por_fold(combinar(p_final, p_hgb, sin_historial, a), folds).mean() for a in ALFAS[1:]}}]
for w in PESOS:
    for grupo in GRUPOS:
        p = especialistas[f"w{w}_{grupo}"]
        filas.append({"w": w, "grupo": grupo, "esp_sin_hist": acc_por_fold(p, folds, sin_historial).mean(),
                      "esp_total": acc_por_fold(p, folds).mean(),
                      **{f"α={a}": acc_por_fold(combinar(p_final, p, sin_historial, a), folds).mean() for a in ALFAS[1:]}})
descriptiva = pd.DataFrame(filas)
print("Referencia: modelo final", acc_final.mean().round(4), "| sin historial",
      acc_por_fold(p_final, folds, sin_historial).mean().round(4))
descriptiva.round(4)

Referencia: modelo final 0.942 | sin historial 0.8931


,w,grupo,esp_sin_hist,esp_total,α=0.25,α=0.5,α=0.75,α=1
0,1,ninguno (P7a),0.8923,0.9412,0.9420,0.9419,0.9418,0.9418
1,2,ninguno,0.8929,0.9407,0.9419,0.9420,0.9418,0.9419
2,2,texto,0.8696,0.9302,0.9420,0.9407,0.9377,0.9348
3,2,vendedor,0.8927,0.9258,0.9422,0.9423,0.9420,0.9419
4,2,fotos,0.8868,0.9386,0.9422,0.9420,0.9408,0.9401
5,2,fechas,0.8784,0.9343,0.9420,0.9414,0.9395,0.9375
6,2,precio,0.8899,0.9395,0.9418,0.9417,0.9415,0.9410
7,3,ninguno,0.8924,0.9403,0.9421,0.9421,0.9421,0.9418
8,3,texto,0.8691,0.9296,0.9420,0.9406,0.9375,0.9347
9,3,vendedor,0.8914,0.9252,0.9420,0.9420,0.9419,0.9415


## 3. Elección anidada y comparación pareada

**Qué hace `seleccion_anidada`:** para cada fold `k`:
1. Con los **otros 4 folds** prueba todas las combinaciones (w, grupo, α) y se queda con la de mayor accuracy total (como solo cambian las filas sin historial, es lo mismo que maximizar ahí). α = 0 (no usar el especialista) va primero: si empata, gana no tocar nada.
2. Aplica esa combinación al fold `k` y mide.

Después, comparación pareada contra el modelo final (pasa si gana 5/5, o 4/5 con media > desvío).

In [4]:
def seleccion_anidada(p_base, esps, folds_, sh):
    candidatos = [("-", "-", 0)] + [(w, g, a) for a in ALFAS[1:] for w in PESOS for g in GRUPOS]
    preds = {c: combinar(p_base, esps[f"w{c[0]}_{c[1]}"], sh, c[2]) >= 0.5 if c[2] else p_base >= 0.5 for c in candidatos}
    acierto = {c: preds[c].astype(int) == y_np for c in candidatos}
    final, elegidos = np.empty(n, bool), []
    for k in range(1, 6):
        entrena = folds_ != k
        mejor = max(candidatos, key=lambda c: acierto[c][entrena].mean())
        elegidos.append(mejor)
        final[folds_ == k] = acierto[mejor][folds_ == k]
    acc = pd.Series(final).groupby(folds_).mean().to_numpy()
    acc_sh = pd.Series(final[sh]).groupby(folds_[sh]).mean().to_numpy()
    return acc, acc_sh, elegidos


acc, acc_sh, elegidos = seleccion_anidada(p_final, especialistas, folds, sin_historial)
resultado = comparar(acc, acc_final)
ref_sh = acc_por_fold(p_final, folds, sin_historial)
print("Elegidos por fold (w, grupo, α):", elegidos)
print("Accuracy por fold:", acc.round(4), "| modelo final:", acc_final.round(4))
print({k: round(float(v), 5) for k, v in resultado.items()})
print(f"Sin historial: {ref_sh.mean():.4f} -> {acc_sh.mean():.4f} (gana {int((np.round(acc_sh - ref_sh, 10) > 0).sum())}/5)")

Elegidos por fold (w, grupo, α): [(2, 'vendedor', 0.5), (2, 'vendedor', 0.5), (2, 'vendedor', 0.5), (2, 'vendedor', 0.5), (2, 'vendedor', 0.5)]
Accuracy por fold: [0.9433 0.9392 0.9426 0.9424 0.9441] | modelo final: [0.9431 0.9389 0.9424 0.942  0.9436]
{'media': 0.94233, 'dif_media': 0.00033, 'dif_desvio': 0.00012, 'folds_gana': 5.0, 'pasa': 1.0}
Sin historial: 0.8931 -> 0.8941 (gana 5/5)


## Robustez: semilla 7 (solo si pasa con la 42)

Se repite todo con otros folds (`random_state=7`): las 18 variantes del especialista con esos folds, "sin historial" redefinido para esos folds y la elección anidada. Referencia: el modelo final con los mismos folds (`oof_final_semilla7.csv`, 0,9422).

In [5]:
if resultado["pasa"]:
    oof7 = pd.read_csv(os.path.join(carpeta_lab, "oof_final_semilla7.csv"))
    folds7 = oof7["fold"].to_numpy()
    p_final7 = 0.5 * oof7["catboost"].to_numpy() + 0.5 * oof7["histgb"].to_numpy()
    sh7 = calcular_sin_historial(folds7)
    especialistas7 = oof_especialistas(folds7, os.path.join(carpeta_lab, "oof_especialista_semilla7.csv"))
    acc7, acc7_sh, elegidos7 = seleccion_anidada(p_final7, especialistas7, folds7, sh7)
    ref7, ref7_sh = acc_por_fold(p_final7, folds7), acc_por_fold(p_final7, folds7, sh7)
    print("Elegidos por fold:", elegidos7)
    print({k: round(float(v), 5) for k, v in comparar(acc7, ref7).items()})
    print(f"Sin historial: {ref7_sh.mean():.4f} -> {acc7_sh.mean():.4f}")
else:
    print("No pasa con la semilla 42: no se corre la semilla 7.")

w2_ninguno       0.7 min


w2_texto         0.2 min


w2_vendedor      0.7 min


w2_fotos         0.7 min


w2_fechas        0.7 min


w2_precio        0.8 min


... (9 outputs left out)

w5_fotos         0.7 min


w5_fechas        0.7 min


w5_precio        0.7 min


Elegidos por fold: [(3, 'vendedor', 0.5), (5, 'vendedor', 0.75), (2, 'vendedor', 0.25), (3, 'vendedor', 0.5), (5, 'ninguno', 0.25)]
{'media': 0.9417, 'dif_media': -0.00046, 'dif_desvio': 0.00032, 'folds_gana': 0.0, 'pasa': 0.0}
Sin historial: 0.8932 -> 0.8917
